In [ ]:
# Copyright (c) Meta Platforms, Inc. and affiliates.

## Video segmentation and tracking with SAM 3

This notebook demonstrates how to use SAM 3 for interactive video segmentation and dense tracking. It covers the following capabilities:

- **Text prompts**: Using natural language descriptions to segment objects (e.g., "person", "shoe")
- **Point prompts**: Adding positive/negative clicks to segment and refine objects

We use the terms _segment_ or _mask_ to refer to the model prediction for an object on a single frame, and _masklet_ to refer to the spatio-temporal masks across the entire video. 

# <a target="_blank" href="https://colab.research.google.com/github/facebookresearch/sam3/blob/main/notebooks/sam3_video_predictor_example.ipynb">
#   <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
# </a>

In [ ]:
using_colab = False

In [ ]:
if using_colab:
    import torch
    import torchvision
    print("PyTorch version:", torch.__version__)
    print("Torchvision version:", torchvision.__version__)
    print("CUDA is available:", torch.cuda.is_available())
    import sys
    !{sys.executable} -m pip install opencv-python matplotlib scikit-learn
    !{sys.executable} -m pip install 'git+https://github.com/facebookresearch/sam3.git'

In [ ]:
!nvidia-smi

## Set-up

In this example, we allow running inference either on a single GPU or multiple GPUs.

In [ ]:
import os
import sam3
import torch

sam3_root = os.path.join(os.path.dirname(sam3.__file__), "..")

# use all available GPUs on the machine
gpus_to_use = range(torch.cuda.device_count())
# # use only a single GPU
# gpus_to_use = [torch.cuda.current_device()]

In [ ]:
from sam3.model_builder import build_sam3_video_predictor

predictor = build_sam3_video_predictor(gpus_to_use=gpus_to_use)

#### Inference and visualization utils

In [ ]:
import glob
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np
from sam3.visualization_utils import (
    load_frame,
    prepare_masks_for_visualization,
    render_masklet_frame,
    visualize_formatted_frame_output,
)

# font size for axes titles
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["figure.titlesize"] = 12


def propagate_in_video(predictor, session_id):
    # propagate from frame 0 to the end of the video
    outputs_per_frame = {}
    for response in predictor.handle_stream_request(
        request=dict(
            type="propagate_in_video",
            session_id=session_id,
        )
    ):
        outputs_per_frame[response["frame_index"]] = response["outputs"]

    return outputs_per_frame


def get_video_fps(video_path, default_fps=10.0):
    # Preserve the source FPS for MP4 input; use a configurable default for JPEG folders.
    if isinstance(video_path, str) and video_path.lower().endswith(".mp4"):
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        cap.release()
        if fps and fps > 0:
            return fps
    return default_fps


def get_foot_point(mask):
    # Use the center of the bottom five mask rows as the person's floor contact point.
    mask = np.squeeze(mask).astype(bool)
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return None

    bottom_y = int(ys.max())
    bottom_xs = xs[ys >= bottom_y - 4]
    return int(np.median(bottom_xs)), bottom_y


def signed_distance_to_line(point, line_start, line_end):
    # Positive and negative values indicate the two sides of the boundary line.
    px, py = point
    x1, y1 = line_start
    x2, y2 = line_end
    line_length = np.hypot(x2 - x1, y2 - y1)
    if line_length == 0:
        raise ValueError("The danger line must contain two different points.")
    return ((x2 - x1) * (py - y1) - (y2 - y1) * (px - x1)) / line_length


def export_tracking_video(
    video_frames,
    outputs_per_frame,
    output_path,
    fps,
    line_start_ratio=(0.15, 0.70),
    line_end_ratio=(0.85, 0.70),
    danger_side=1,
    line_margin=8,
    confirm_frames=3,
    alpha=0.5,
):
    # Render masks and warn when a tracked person's foot crosses the line.
    if not video_frames:
        raise ValueError("No video frames were loaded.")

    first_frame = load_frame(video_frames[0])
    height, width = first_frame.shape[:2]
    line_start = (
        int(line_start_ratio[0] * width),
        int(line_start_ratio[1] * height),
    )
    line_end = (
        int(line_end_ratio[0] * width),
        int(line_end_ratio[1] * height),
    )
    object_states = {}
    intrusion_events = []
    writer = cv2.VideoWriter(
        output_path,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height),
    )
    if not writer.isOpened():
        raise RuntimeError(f"Could not open output video: {output_path}")

    try:
        for frame_idx in sorted(outputs_per_frame):
            rgb_frame = load_frame(video_frames[frame_idx])
            frame_output = outputs_per_frame[frame_idx]
            rendered_frame = render_masklet_frame(
                rgb_frame,
                frame_output,
                frame_idx=frame_idx,
                alpha=alpha,
            )

            active_intrusions = []
            for object_id, mask in zip(
                frame_output["out_obj_ids"],
                frame_output["out_binary_masks"],
            ):
                object_id = int(object_id)
                foot_point = get_foot_point(mask)
                if foot_point is None:
                    continue

                distance = signed_distance_to_line(
                    foot_point, line_start, line_end
                )
                if distance > line_margin:
                    current_side = 1
                elif distance < -line_margin:
                    current_side = -1
                else:
                    current_side = 0

                state = object_states.setdefault(
                    object_id,
                    {
                        "last_stable_side": current_side,
                        "danger_frames": 0,
                        "inside": current_side == danger_side,
                    },
                )

                if current_side == danger_side:
                    if state["last_stable_side"] == -danger_side:
                        state["danger_frames"] += 1
                        if state["danger_frames"] >= confirm_frames:
                            state["inside"] = True
                            state["last_stable_side"] = danger_side
                            intrusion_events.append(
                                {
                                    "frame_idx": frame_idx,
                                    "object_id": object_id,
                                    "foot_point": foot_point,
                                }
                            )
                            print(
                                f"[DANGER ZONE ENTRY] frame={frame_idx}, "
                                f"person_id={object_id}"
                            )
                elif current_side == -danger_side:
                    state["last_stable_side"] = -danger_side
                    state["danger_frames"] = 0
                    state["inside"] = False

                foot_color = (255, 0, 0) if state["inside"] else (0, 255, 0)
                cv2.circle(rendered_frame, foot_point, 7, foot_color, -1)
                if state["inside"]:
                    active_intrusions.append(object_id)

            cv2.line(rendered_frame, line_start, line_end, (255, 255, 0), 4)
            cv2.putText(
                rendered_frame,
                "DANGER ZONE",
                (line_start[0], max(35, line_start[1] - 15)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.9,
                (255, 255, 0),
                2,
                cv2.LINE_AA,
            )

            if active_intrusions:
                warning = "DANGER! PERSON " + ", ".join(map(str, active_intrusions))
                cv2.rectangle(rendered_frame, (15, 15), (width - 15, 75), (255, 0, 0), -1)
                cv2.putText(
                    rendered_frame,
                    warning,
                    (30, 58),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.1,
                    (255, 255, 255),
                    3,
                    cv2.LINE_AA,
                )

            writer.write(cv2.cvtColor(rendered_frame, cv2.COLOR_RGB2BGR))
    finally:
        writer.release()

    print(f"Saved tracking video to: {os.path.abspath(output_path)}")
    print(f"Detected {len(intrusion_events)} danger-zone entries.")
    return intrusion_events


### Loading an example video

We assume that the video is stored as either **a list of JPEG frames with filenames like `<frame_index>.jpg`** or **an MP4 video**.

Note that you can extract their JPEG frames using ffmpeg (https://ffmpeg.org/) as follows:
```
ffmpeg -i <your_video>.mp4 -q:v 2 -start_number 0 <output_dir>/'%05d.jpg'
```
where `-q:v` generates high-quality JPEG frames and `-start_number 0` asks ffmpeg to start the JPEG file from `00000.jpg`.

In [ ]:
# "video_path" needs to be either a JPEG folder or a MP4 video file
video_path = f"{sam3_root}/assets/videos/factory_2.mp4"

In [ ]:
# load "video_frames_for_vis" for visualization purposes (they are not used by the model)
if isinstance(video_path, str) and video_path.endswith(".mp4"):
    cap = cv2.VideoCapture(video_path)
    video_frames_for_vis = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        video_frames_for_vis.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
else:
    video_frames_for_vis = glob.glob(os.path.join(video_path, "*.jpg"))
    try:
        # integer sort instead of string sort (so that e.g. "2.jpg" is before "11.jpg")
        video_frames_for_vis.sort(
            key=lambda p: int(os.path.splitext(os.path.basename(p))[0])
        )
    except ValueError:
        # fallback to lexicographic sort if the format is not "<frame_index>.jpg"
        print(
            f'frame names are not in "<frame_index>.jpg" format: {video_frames_for_vis[:5]=}, '
            f"falling back to lexicographic sort."
        )
        video_frames_for_vis.sort()

### Opening an inference session on this video

SAM 3 requires stateful inference for interactive video segmentation, so we need to initialize an **inference session** on this video.

During initialization, it loads all the video frames and stores their pixels in the session state.

In [ ]:
response = predictor.handle_request(
    request=dict(
        type="start_session",
        resource_path=video_path,
    )
)
session_id = response["session_id"]

### Video promptable concept segmentation with text

Using SAM 3 you can describe objects using natural language, and the model will automatically detect and track all instances of that object throughout the video.

In the example below, we add a text prompt on frame 0 and propagation throughout the video. Here we use the text prompt "person" to detect all people in the video. SAM 3 will automatically identify multiple person instances and assign each a unique object ID.

Note that the first call might be slower due to setting up buffers. **You can rerun all the cells below when measuring speed.**

In [ ]:
# note: in case you already ran one text prompt and now want to switch to another text prompt
# it's required to reset the session first (otherwise the results would be wrong)
_ = predictor.handle_request(
    request=dict(
        type="reset_session",
        session_id=session_id,
    )
)

In [ ]:
prompt_text_str = "person"
frame_idx = 0  # add a text prompt on frame 0
response = predictor.handle_request(
    request=dict(
        type="add_prompt",
        session_id=session_id,
        frame_index=frame_idx,
        text=prompt_text_str,
    )
)
out = response["outputs"]

plt.close("all")
visualize_formatted_frame_output(
    frame_idx,
    video_frames_for_vis,
    outputs_list=[prepare_masks_for_visualization({frame_idx: out})],
    titles=["SAM 3 Dense Tracking outputs"],
    figsize=(6, 4),
)

In [ ]:
# now we propagate the outputs from frame 0 to the end of the video and collect all outputs
outputs_per_frame = propagate_in_video(predictor, session_id)

# export the tracking result as an MP4 before formatting mutates the output dictionary
output_video_path = "sam3_person_tracking_danger_zone.mp4"
intrusion_events = export_tracking_video(
    video_frames_for_vis,
    outputs_per_frame,
    output_video_path,
    fps=get_video_fps(video_path),
    # Normalized (x, y) coordinates: edit these two points to move the line.
    line_start_ratio=(0.15, 0.70),
    line_end_ratio=(0.85, 0.70),
    # 1 means the side below this left-to-right line is dangerous; use -1 to reverse.
    danger_side=1,
    line_margin=8,
    confirm_frames=3,
)

# finally, we reformat the outputs for visualization and plot the outputs every 60 frames
outputs_per_frame = prepare_masks_for_visualization(outputs_per_frame)

vis_frame_stride = 60
plt.close("all")
for frame_idx in range(0, len(outputs_per_frame), vis_frame_stride):
    visualize_formatted_frame_output(
        frame_idx,
        video_frames_for_vis,
        outputs_list=[outputs_per_frame],
        titles=["SAM 3 Dense Tracking outputs"],
        figsize=(6, 4),
    )

### Close session

Each session is tied to a single video. We can close the session after inference to free up its resources.

(Then, you may start a new session on another video.)

In [ ]:
# finally, close the inference session to free its GPU resources
# (you may start a new session on another video)
_ = predictor.handle_request(
    request=dict(
        type="close_session",
        session_id=session_id,
    )
)

### Clean-up

After all inference is done, we can shutdown the predictor to free up the multi-GPU process group.

In [ ]:
# after all inference is done, we can shutdown the predictor
# to free up the multi-GPU process group
predictor.shutdown()